In [1]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import GroupKFold
from scipy.io import loadmat

# ─────────────────────────────────────────────────────────────────────────────
# LABEL MAPPING — must match main.py exactly
#
#   main.py class_names:  0=Meningioma  1=Glioma  2=Pituitary Tumor  3=Normal
#
#   Dataset original labels:
#       1 = Meningioma  →  remap to 0
#       2 = Glioma      →  remap to 1
#       3 = Pituitary   →  remap to 2
#       4 = Normal      →  remap to 3
# ─────────────────────────────────────────────────────────────────────────────
LABEL_REMAP = {1: 0, 2: 1, 3: 2, 4: 3}

CLASS_NAMES = {
    0: "Meningioma",
    1: "Glioma",
    2: "Pituitary Tumor",
    3: "Normal",
}

# Print order for per-class breakdown
CLASS_PRINT_ORDER = [0, 1, 2, 3]

# ─────────────────────────────────────────────────────────────────────────────
# CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────
NUM_ORIGINALS = 2832
NUM_CLASSES   = 4
N_FOLDS       = 5
EPOCHS        = 30
BATCH         = 16
LR            = 1e-4
DEVICE        = "mps" if torch.backends.mps.is_available() else "cpu"

mat_dir = "/Users/ha/Documents/School File/Final Year project/Dataset/brainTumorDataPublic/preprocessed_dynamic_per_image"


# ─────────────────────────────────────────────────────────────────────────────
# MODEL — DenseNetWithSegmentation
# Identical architecture to main.py so the saved .pth loads directly
# ─────────────────────────────────────────────────────────────────────────────
class DenseNetWithSegmentation(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()

        base = models.densenet121(weights="IMAGENET1K_V1")   # pretrained features

        # Unfreeze all — fine-tune the whole network
        for param in base.parameters():
            param.requires_grad = True

        self.features = base.features   # output: (B, 1024, 7, 7) for 224×224 input

        # Classification head  (matches main.py exactly)
        self.classifier = nn.Sequential(
            nn.Linear(base.classifier.in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )

        # Segmentation head  (matches main.py exactly)
        self.seg_head = nn.Sequential(
            nn.Conv2d(1024, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),   # 7→14
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),   # 14→28
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),   # 28→56
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),   # 56→112
            nn.Conv2d(32, 1, kernel_size=1),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),   # 112→224
            nn.Sigmoid(),
        )

    def forward(self, x):
        features   = F.relu(self.features(x), inplace=True)
        pooled     = F.adaptive_avg_pool2d(features, (1, 1))
        cls_output = self.classifier(torch.flatten(pooled, 1))
        seg_output = self.seg_head(features)   # (B, 1, 224, 224)
        return cls_output, seg_output


# ─────────────────────────────────────────────────────────────────────────────
# DATASET
# ─────────────────────────────────────────────────────────────────────────────
class TumorDataset(Dataset):
    def __init__(self, mat_dir, transform=None):
        self.mat_dir   = mat_dir
        self.transform = transform
        self.files     = sorted(
            [f for f in os.listdir(mat_dir) if f.endswith(".mat")]
        )

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        filepath = os.path.join(self.mat_dir, self.files[idx])
        mat_data = loadmat(filepath)
        cj       = mat_data["cjdata"][0][0]

        img   = np.array(cj["image"]).astype(np.float32)
        label = int(np.array(cj["label"]).item())

        # Remap to match main.py class_names indices
        label = LABEL_REMAP[label]

        # Min-max normalise
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)

        # Grayscale → 3-channel
        img = np.stack([img, img, img], axis=-1)

        if self.transform:
            img = self.transform(img)

        return img, label


# ─────────────────────────────────────────────────────────────────────────────
# GROUP ARRAY  (prevents data leakage across augmented variants)
# ─────────────────────────────────────────────────────────────────────────────
def build_groups(file_list, num_originals=NUM_ORIGINALS):
    groups = []
    for fname in file_list:
        number = int(os.path.splitext(fname)[0])
        groups.append((number - 1) % num_originals)
    return np.array(groups)


# ─────────────────────────────────────────────────────────────────────────────
# METRICS
# ─────────────────────────────────────────────────────────────────────────────
def per_class_stats(y_true, y_pred, num_classes=NUM_CLASSES):
    cm = confusion_matrix(y_true, y_pred, labels=range(num_classes))
    stats = {}
    for i in range(num_classes):
        tp = int(cm[i, i])
        fp = int(cm[:, i].sum() - tp)
        fn = int(cm[i, :].sum() - tp)
        tn = int(cm.sum() - tp - fp - fn)
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        f1   = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0
        stats[i] = dict(tp=tp, tn=tn, fp=fp, fn=fn,
                        precision=prec, recall=rec, specificity=spec, f1=f1)
    return stats


def compute_overall(y_true, y_pred, num_classes=NUM_CLASSES):
    acc  = 100 * np.mean(np.array(y_pred) == np.array(y_true))
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    cm   = confusion_matrix(y_true, y_pred, labels=range(num_classes))
    specs, support = [], []
    for i in range(num_classes):
        tp = cm[i, i]; fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp; tn = cm.sum() - tp - fp - fn
        specs.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
        support.append(cm[i, :].sum())
    spec = float(np.average(specs, weights=support))
    return acc, prec, rec, f1, spec


def print_per_class(stats, fold_label=""):
    sep = "=" * 66
    print(f"\n{sep}")
    if fold_label:
        print(f"  PER-CLASS BREAKDOWN  —  {fold_label}")
    else:
        print(f"  PER-CLASS BREAKDOWN  (Averaged Across All {N_FOLDS} Folds)")
    print(sep)
    for idx in CLASS_PRINT_ORDER:
        name = CLASS_NAMES[idx]
        s    = stats[idx]
        pad  = max(0, 44 - len(name))
        print(f"\n  ┌─ Class : {name} {'─' * pad}┐")
        print(f"  │  TP  – Correctly identified as {name:<18}: {s['tp']:>7.1f}  │")
        print(f"  │  TN  – Correctly rejected  as  {name:<18}: {s['tn']:>7.1f}  │")
        print(f"  │  FP  – Wrongly flagged      as  {name:<18}: {s['fp']:>7.1f}  │")
        print(f"  │  FN  – Missed, actually is  {name:<18}: {s['fn']:>7.1f}  │")
        print(f"  │                                                              │")
        print(f"  │  Precision   : {s['precision']*100:6.2f}%                                  │")
        print(f"  │  Recall      : {s['recall']*100:6.2f}%                                  │")
        print(f"  │  Specificity : {s['specificity']*100:6.2f}%                                  │")
        print(f"  │  F1-Score    : {s['f1']*100:6.2f}%                                  │")
        print(f"  └{'─' * 62}┘")


# ─────────────────────────────────────────────────────────────────────────────
# TRAIN / EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
def run_epoch(model, loader, optimizer, criterion, device, training=True):
    model.train() if training else model.eval()
    total_loss, all_labels, all_preds = 0.0, [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if training:
                optimizer.zero_grad()

            cls_output, _ = model(imgs)   # ignore seg output during training
            loss = criterion(cls_output, labels)

            if training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(cls_output, 1)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())

    return total_loss / len(loader), all_labels, all_preds


# ─────────────────────────────────────────────────────────────────────────────
# MAIN — GroupKFold cross-validation
# ─────────────────────────────────────────────────────────────────────────────
# ImageNet normalisation — matches main.py's model_transform
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224, 224)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

full_dataset = TumorDataset(mat_dir, transform)
groups       = build_groups(full_dataset.files)
indices      = np.arange(len(full_dataset))

print(f"Total samples : {len(full_dataset)}")
print(f"Unique groups : {len(np.unique(groups))}  (= original images)")
print(f"Label mapping : {LABEL_REMAP}  →  matches main.py class_names")
print(f"Running {N_FOLDS}-Fold GroupKFold — no original appears in both train and val\n")

gkf          = GroupKFold(n_splits=N_FOLDS)
fold_results = []

accumulated = {
    i: {k: [] for k in ["tp", "tn", "fp", "fn", "precision", "recall", "specificity", "f1"]}
    for i in range(NUM_CLASSES)
}

for fold, (train_idx, val_idx) in enumerate(gkf.split(indices, groups=groups), 1):

    # Sanity check — zero group overlap
    train_groups = set(groups[train_idx])
    val_groups   = set(groups[val_idx])
    assert len(train_groups & val_groups) == 0, \
        f"Fold {fold}: DATA LEAKAGE — {len(train_groups & val_groups)} shared groups!"

    print(f"\n{'='*66}")
    print(f"  FOLD {fold} / {N_FOLDS}   |  Train: {len(train_idx)} samples  |  Val: {len(val_idx)} samples")
    print(f"{'='*66}")

    train_loader = DataLoader(
        Subset(full_dataset, train_idx), batch_size=BATCH, shuffle=True,  num_workers=0
    )
    val_loader = DataLoader(
        Subset(full_dataset, val_idx),   batch_size=BATCH, shuffle=False, num_workers=0
    )

    model     = DenseNetWithSegmentation(num_classes=NUM_CLASSES).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_f1, best_lbl, best_prd = 0.0, None, None

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_lbl, tr_prd = run_epoch(model, train_loader, optimizer, criterion, DEVICE, training=True)
        vl_loss, vl_lbl, vl_prd = run_epoch(model, val_loader,   optimizer, criterion, DEVICE, training=False)
        scheduler.step()

        tr_acc, tr_p, tr_r, tr_f1, tr_sp = compute_overall(tr_lbl, tr_prd)
        vl_acc, vl_p, vl_r, vl_f1, vl_sp = compute_overall(vl_lbl, vl_prd)

        print(
            f"  Ep {epoch:03d}/{EPOCHS}  "
            f"TrLoss {tr_loss:.4f}  TrAcc {tr_acc:.1f}%  TrF1 {tr_f1:.3f}  "
            f"| ValLoss {vl_loss:.4f}  ValAcc {vl_acc:.1f}%  ValF1 {vl_f1:.3f}  ValSpec {vl_sp:.3f}"
        )

        if vl_f1 > best_f1:
            best_f1  = vl_f1
            best_lbl = vl_lbl[:]
            best_prd = vl_prd[:]
            # Save only the state_dict so main.py can load it directly
            torch.save(model.state_dict(), f"best_model_fold{fold}.pth")
            print(f"    >> New best val F1 = {best_f1:.4f}  (saved as best_model_fold{fold}.pth)")

    # Per-class breakdown for this fold
    fold_stats = per_class_stats(best_lbl, best_prd, num_classes=NUM_CLASSES)
    print_per_class(fold_stats, fold_label=f"Fold {fold}")

    for cls_idx, s in fold_stats.items():
        for metric, val in s.items():
            accumulated[cls_idx][metric].append(val)

    vl_acc, vl_p, vl_r, vl_f1, vl_sp = compute_overall(best_lbl, best_prd)
    fold_results.append(dict(fold=fold, acc=vl_acc, prec=vl_p,
                             rec=vl_r, f1=vl_f1, spec=vl_sp))


# ─────────────────────────────────────────────────────────────────────────────
# FINAL RESULTS — print fold table THEN averaged per-class boxes
# Matches screenshot layout: table on top, per-class boxes below
# ─────────────────────────────────────────────────────────────────────────────
LINE = "=" * 68

# ── Fold-by-fold summary table (top section) ─────────────────────────────────
print(f"\n{LINE}")
print(f"  FINAL RESULTS — AUGMENTED")
print(LINE)
print(f"  {'Fold':<8} {'Accuracy':>10} {'Precision':>10} {'Recall':>8} {'Specificity':>12} {'F1-Score':>10}")
print(f"  {'-'*64}")
for r in fold_results:
    print(f"  Fold_{r['fold']}   {r['acc']/100:>10.6f}  {r['prec']:>10.6f}  "
          f"{r['rec']:>8.6f}  {r['spec']:>12.6f}  {r['f1']:>10.6f}")
print(f"  {'-'*64}")

mean_acc  = np.mean([r['acc']  for r in fold_results]) / 100
mean_prec = np.mean([r['prec'] for r in fold_results])
mean_rec  = np.mean([r['rec']  for r in fold_results])
mean_spec = np.mean([r['spec'] for r in fold_results])
mean_f1   = np.mean([r['f1']   for r in fold_results])

print(f"\n  MEAN ACCURACY    : {mean_acc:.4f}")
print(f"  MEAN PRECISION   : {mean_prec:.4f}")
print(f"  MEAN RECALL      : {mean_rec:.4f}")
print(f"  MEAN SPECIFICITY : {mean_spec:.4f}")
print(f"  MEAN F1-SCORE    : {mean_f1:.4f}")
print(LINE)

# ── Averaged per-class breakdown (bottom section) ─────────────────────────────
avg_stats = {
    cls_idx: {metric: float(np.mean(vals)) for metric, vals in metrics.items()}
    for cls_idx, metrics in accumulated.items()
}
print_per_class(avg_stats)   # prints "Averaged Across All 5 Folds"

print(f"\n>> To use on your website: copy the best fold's best_model_foldX.pth")
print(f"   and rename it to best_model.pth in your tumor_project/ folder.")

Total samples : 16992
Unique groups : 2832  (= original images)
Label mapping : {1: 0, 2: 1, 3: 2, 4: 3}  →  matches main.py class_names
Running 5-Fold GroupKFold — no original appears in both train and val


  FOLD 1 / 5   |  Train: 13590 samples  |  Val: 3402 samples
  Ep 001/30  TrLoss 0.2041  TrAcc 92.6%  TrF1 0.926  | ValLoss 0.0850  ValAcc 97.1%  ValF1 0.971  ValSpec 0.990
    >> New best val F1 = 0.9708  (saved as best_model_fold1.pth)
  Ep 002/30  TrLoss 0.0656  TrAcc 97.9%  TrF1 0.979  | ValLoss 0.0549  ValAcc 98.1%  ValF1 0.981  ValSpec 0.994
    >> New best val F1 = 0.9814  (saved as best_model_fold1.pth)
  Ep 003/30  TrLoss 0.0417  TrAcc 98.5%  TrF1 0.985  | ValLoss 0.0512  ValAcc 98.2%  ValF1 0.982  ValSpec 0.994
    >> New best val F1 = 0.9821  (saved as best_model_fold1.pth)
  Ep 004/30  TrLoss 0.0260  TrAcc 99.1%  TrF1 0.991  | ValLoss 0.0506  ValAcc 98.4%  ValF1 0.984  ValSpec 0.995
    >> New best val F1 = 0.9835  (saved as best_model_fold1.pth)
  Ep 005/30  TrLoss 0.